# Soluciones — Clase 25: Ajuste de Hiperparámetros y Selección de Modelos

> Este notebook contiene las soluciones completas a todos los ejercicios.
> Intenta resolverlos por tu cuenta antes de consultar aquí.

## Solución Ejercicio 1 — Identificar parámetros vs hiperparámetros

| Elemento | Tipo | Explicación |
|---|---|---|
| Coeficientes de regresión lineal | **Parámetro** | El modelo los aprende minimizando el error |
| `max_depth=5` en árbol | **Hiperparámetro** | Lo definimos nosotros antes de entrenar |
| Pesos de red neuronal | **Parámetro** | Se aprenden con backpropagation |
| `n_neighbors=7` en KNN | **Hiperparámetro** | Lo definimos nosotros |
| Umbral de división del árbol | **Parámetro** | El algoritmo lo determina buscando la mejor división |
| `C=0.1` en SVM | **Hiperparámetro** | Lo definimos nosotros (penalización por error) |
| `n_estimators=200` en RF | **Hiperparámetro** | Lo definimos nosotros |
| Probabilidades en Naive Bayes | **Parámetro** | Se calculan a partir de los datos de entrenamiento |

In [ ]:
# Solución Ejercicio 2 — Modelo base
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, validation_curve, learning_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from scipy.stats import randint

df = pd.read_csv('../../datasets/estudiantes.csv')
X = df.drop(columns=['aprobado'])
y = df['aprobado']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo_base = RandomForestClassifier(random_state=42)
modelo_base.fit(X_train, y_train)

acc_base = accuracy_score(y_test, modelo_base.predict(X_test))
print(f'Accuracy base: {acc_base:.4f}')
print(f'n_estimators por defecto: {modelo_base.n_estimators}')
print(f'max_depth por defecto: {modelo_base.max_depth}  <- None significa sin límite (puede sobreajustar)')

In [ ]:
# Solución Ejercicio 3 — GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None]
}
# 3 x 4 = 12 combinaciones
# Con cv=5: 12 x 5 = 60 modelos entrenados
print(f'Combinaciones: {3 * 4}')
print(f'Modelos entrenados total: {3 * 4 * 5}')

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print(f'\nMejores parámetros: {grid_search.best_params_}')
print(f'Mejor score CV: {grid_search.best_score_:.4f}')
print(f'Score en test: {grid_search.score(X_test, y_test):.4f}')
print(f'Mejora vs base: +{(grid_search.score(X_test, y_test) - acc_base)*100:.2f}%')

In [ ]:
# Solución Ejercicio 4 — Explorar resultados del grid
resultados = pd.DataFrame(grid_search.cv_results_)

top5 = resultados.sort_values('rank_test_score')[
    ['params', 'mean_test_score', 'std_test_score', 'rank_test_score']
].head(5)

print('Top 5 combinaciones:')
print(top5.to_string())

# La std_test_score importa: queremos el modelo más estable (menor desviación)
# Un modelo con score ligeramente menor pero std mucho menor puede ser preferible
print('\nNota: considera elegir el modelo con menor std_test_score si hay scores similares.')
print('Un modelo más estable (baja varianza) es más confiable en producción.')

In [ ]:
# Solución Ejercicio 5 — RandomizedSearchCV
param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(2, 15),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10)
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train, y_train)

print(f'Mejores parámetros: {random_search.best_params_}')
print(f'Mejor score CV: {random_search.best_score_:.4f}')
print(f'Score en test: {random_search.score(X_test, y_test):.4f}')
print(f'\nComparación:')
print(f'  Base:              {acc_base:.4f}')
print(f'  GridSearchCV:      {grid_search.score(X_test, y_test):.4f}')
print(f'  RandomizedSearch:  {random_search.score(X_test, y_test):.4f}')
print('\nUsa RandomizedSearchCV cuando el espacio de búsqueda es grande (muchos hiperparámetros).')
print('Generalmente encuentra soluciones muy similares a GridSearch en mucho menos tiempo.')

In [ ]:
# Solución Ejercicio 6 — Curva de validación
param_range = [2, 5, 10, 15, 20, 30, 50]

train_scores_vc, val_scores_vc = validation_curve(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_train, y_train,
    param_name='max_depth',
    param_range=param_range,
    cv=5,
    scoring='accuracy'
)

plt.figure(figsize=(10, 5))
plt.plot(param_range, train_scores_vc.mean(axis=1), 'o-', color='steelblue', label='Entrenamiento')
plt.fill_between(param_range,
                 train_scores_vc.mean(axis=1) - train_scores_vc.std(axis=1),
                 train_scores_vc.mean(axis=1) + train_scores_vc.std(axis=1),
                 alpha=0.15, color='steelblue')
plt.plot(param_range, val_scores_vc.mean(axis=1), 'o-', color='darkorange', label='Validación')
plt.fill_between(param_range,
                 val_scores_vc.mean(axis=1) - val_scores_vc.std(axis=1),
                 val_scores_vc.mean(axis=1) + val_scores_vc.std(axis=1),
                 alpha=0.15, color='darkorange')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Curva de validación: efecto de max_depth')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Encontrar el max_depth óptimo
val_means = val_scores_vc.mean(axis=1)
idx_optimo = np.argmax(val_means)
print(f'max_depth óptimo según curva de validación: {param_range[idx_optimo]}')
print(f'Score de validación en ese punto: {val_means[idx_optimo]:.4f}')
print('\nSe ve overfitting cuando la curva de TRAIN sigue subiendo pero la de VALIDACION baja.')

In [ ]:
# Solución Ejercicio 7 — Modelo final optimizado
modelo_final = RandomForestClassifier(**grid_search.best_params_, random_state=42)
modelo_final.fit(X_train, y_train)

y_pred = modelo_final.predict(X_test)
acc_final = accuracy_score(y_test, y_pred)

print('=== Evaluación final del modelo optimizado ===')
print(classification_report(y_test, y_pred))

mejora = (acc_final - acc_base) * 100
print(f'Accuracy base:       {acc_base:.4f}')
print(f'Accuracy optimizado: {acc_final:.4f}')
print(f'Mejora:              +{mejora:.2f}%')

if mejora > 0:
    print(f'\nEl ajuste de hiperparámetros mejoró el modelo un {mejora:.2f}%.')
else:
    print('\nEn este caso el ajuste no mejoró significativamente.')
    print('Esto puede pasar cuando el modelo base ya era casi óptimo,') 
    print('o cuando el dataset es pequeño y la varianza domina.')